In [2]:
# 02 - CNN Baseline
# Trains a simple from-scratch CNN on data/processed/ (created by 01_data_prep.ipynb)

import sys
sys.path.insert(0, '..')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models.cnn_model import SimpleCNN

In [5]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
# make sure the split from step 1 actually exists before going further
train_dir = "../data/processed/train"
val_dir = "../data/processed/val"

if not os.path.isdir(train_dir):
    print("train folder not found, run 01_data_prep.ipynb first")

In [7]:
# basic image settings for a first test run
image_size = 128
batch_size = 16
epochs = 10
learning_rate = 0.0001

# simple transform - resize and convert to tensor
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

In [8]:
# load train and val data using folder names as class labels
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

# check the class order matches what we expect
print(train_data.class_to_idx)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

{'healthy': 0, 'low_tread': 1, 'sidewall_damaged': 2, 'uneven_wear': 3, 'zero_tread': 4}


In [9]:
# build the model
num_classes = len(train_data.classes)
model = SimpleCNN(num_classes).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [10]:
# training loop - simple version, one pass through train + val each epoch
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total

    # check performance on val set
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total

    print("Epoch", epoch + 1, "- train loss:", round(train_loss, 3),
          "train acc:", round(train_accuracy, 3),
          "val acc:", round(val_accuracy, 3))

Epoch 1 - train loss: 20.82 train acc: 0.23 val acc: 0.333
Epoch 2 - train loss: 20.589 train acc: 0.333 val acc: 0.333
Epoch 3 - train loss: 20.285 train acc: 0.333 val acc: 0.333
Epoch 4 - train loss: 19.67 train acc: 0.343 val acc: 0.333
Epoch 5 - train loss: 19.359 train acc: 0.348 val acc: 0.333
Epoch 6 - train loss: 19.238 train acc: 0.358 val acc: 0.333
Epoch 7 - train loss: 19.345 train acc: 0.333 val acc: 0.333
Epoch 8 - train loss: 19.036 train acc: 0.358 val acc: 0.333
Epoch 9 - train loss: 18.967 train acc: 0.333 val acc: 0.381
Epoch 10 - train loss: 19.155 train acc: 0.343 val acc: 0.381


In [ ]:
# save the trained model
os.makedirs("../results/cnn", exist_ok=True)
torch.save(model.state_dict(), "../results/cnn/model.pt")
print("Model saved to results/cnn/model.pt")